In [2]:
from ucimlrepo import fetch_ucirepo
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import plotly.express as px


In [3]:
wine_quality = fetch_ucirepo(name='Wine Quality')
wine_quality_data = wine_quality['data']['original']

In [4]:
seed = 42 #if it must be 42 :P

wine_X = wine_quality_data.drop(columns = ['quality']) # drop quality
wine_X['color'] = (wine_X['color'] == "red").astype(float) # make color a number instead of a string
scaler = StandardScaler()
wine_X = scaler.fit_transform(wine_X) # scaling values
num_features = wine_X.shape[1]

wine_y = wine_quality_data['quality']
min_quality = wine_y.min()
num_classes = wine_y.max() - min_quality + 1
wine_y -= min_quality

wine_X_train_val, wine_X_test, wine_y_train_val, wine_y_test = train_test_split(
    wine_X, wine_y, test_size=0.2, random_state=42
)

wine_X_tr, wine_X_val, wine_y_tr, wine_y_val = train_test_split(
    wine_X_train_val, wine_y_train_val, test_size=0.25, random_state=42
)
print(wine_X_tr.shape, wine_X_val.shape, wine_y_tr.shape, wine_y_val.shape)


(3897, 12) (1300, 12) (3897,) (1300,)


In [5]:
# this makes them tensors for pytorch

wine_X_tr = torch.tensor(wine_X_tr, dtype=torch.float32)
wine_X_val = torch.tensor(wine_X_val, dtype=torch.float32)
wine_X_test = torch.tensor(wine_X_test, dtype=torch.float32)

wine_y_tr = torch.tensor(wine_y_tr, dtype=torch.long)
wine_y_val = torch.tensor(wine_y_val.values, dtype=torch.long)
wine_y_test = torch.tensor(wine_y_test.values, dtype=torch.long)

In [6]:
dataset = torch.utils.data.TensorDataset(wine_X_tr, wine_y_tr)
trainloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)


In [13]:
from typing import Any


# this makes a model given the hidden layers. Just MLP model with RELU activation
def get_model(hidden_dims):
    layer_dims = [num_features, *hidden_dims, num_classes]
    layers = []
    for i in range(len(layer_dims)-2):
        layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))
    return nn.Sequential(*layers)

# This evaluates accuracy, kinda like sklearn.score
def accuracy(model, features, labels):
    with torch.no_grad():
        return (torch.argmax(model(features), dim=1) == labels).sum().item() / len(labels)

# this trains a model
def train(epochs, **kwargs):
    hidden_dims = kwargs.pop("hidden_dims", [16, 32, 16])
    print(f"training with hidden_dims: {hidden_dims}, epochs: {epochs}, and kwargs: {kwargs}")

    model = get_model(hidden_dims)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), **kwargs)

    train_accuracy = []
    val_accuracy = []

    for epoch in range(epochs):
        model.train()
        for _, data in enumerate(trainloader, 0):
            inputs, batch_labels = data
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

        model.eval()
        train_accuracy.append((epoch, accuracy(model, wine_X_tr, wine_y_tr)))
        val_accuracy.append((epoch, accuracy(model, wine_X_val, wine_y_val)))

    return train_accuracy, val_accuracy

def plotly_plot(lines: list[tuple[list[tuple[Any, Any]], str]], x_name: str = "x axis", y_name: str = "y axis", label_name: str = "label", **kwargs):
    df = pd.DataFrame([{x_name: x, y_name: y, label_name: label} for data, label in lines for x, y in data])
    fig = px.line(df, x=x_name, y=y_name, color=label_name, **kwargs)
    fig.show()

def train_and_plot(params_list: list[dict], param_to_test: str, epochs=300):
    train_accuracies = []
    val_accuracies = []
    for params in params_list:
        train_accuracy, val_accuracy = train(epochs, **params)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
    plotly_plot(
        list(zip(train_accuracies, [str(params[param_to_test]) for params in params_list])), "epoch", "training accuracy", param_to_test
    )
    plotly_plot(
        list(zip(val_accuracies, [str(params[param_to_test]) for params in params_list])), "epoch", "validation accuracy", param_to_test
    )

In [14]:
train_and_plot(
    [
        {"hidden_dims": [16]},
        {"hidden_dims": [32]},
        {"hidden_dims": [64]},
        {"hidden_dims": [16, 32, 16]},
        {"hidden_dims": [16, 64, 16]},
        {"hidden_dims": [32, 64, 32]}
    ],
    "hidden_dims"
)

training with hidden_dims: [16], epochs: 300, and kwargs: {}
training with hidden_dims: [32], epochs: 300, and kwargs: {}
training with hidden_dims: [64], epochs: 300, and kwargs: {}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {}
training with hidden_dims: [16, 64, 16], epochs: 300, and kwargs: {}
training with hidden_dims: [32, 64, 32], epochs: 300, and kwargs: {}


In [15]:
train_and_plot(
    [
        {"lr": 0.0001},
        {"lr": 0.0005},
        {"lr": 0.001},
        {"lr": 0.005},
        {"lr": 0.01},
        {"lr": 0.05},
        {"lr": 0.1},
        {"lr": 0.5},
        {"lr": 1},
    ],
    "lr"
)

training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.0001}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.0005}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.001}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.005}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.01}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.05}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.1}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.5}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 1}
